# Synthetic data and domain randomization in Isaac Sim

**Time:** 2–4 hours (Isaac Sim + Cosmos GPU)  
**Prerequisites:** notebooks 04–06 and a working full pipeline.

## Learning objectives

- Generate reproducible scene variations rather than random chaos
- Vary object layout, yaw, surface appearance, and lighting one factor at a time
- Keep a held-out test set and compare robustness with notebook 04 metrics
- Reject synthetic scenes that violate the physical cell

> Domain randomization is useful only when the range represents plausible real conditions. It does not repair bad calibration or replace physical validation.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path
import matplotlib.pyplot as plt

def find_root():
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p / 'fanuc_er4ia_sim' / 'scene_config.py').is_file(): return p
    raise FileNotFoundError('Repository root not found')
ROOT = find_root(); SIM = ROOT / 'fanuc_er4ia_sim'; RUNS = SIM / 'runs'
sys.path.insert(0, str(SIM))
import scene_config as C
print('Baseline layout seed:', C.CUBE_LAYOUT_SEED)
print('Baseline cubes:', C.SORT_CUBES)
print('Baseline yaws:', C.SORT_CUBE_YAWS_DEG)

## 1. Define a small experiment matrix

Start with a baseline and one controlled factor. Good first factors are three valid layout seeds and several cube yaws. Later, test plausible light intensity and paper roughness ranges. Do not randomize camera calibration, cube size, gripper dimensions, or table height unless the experiment is specifically about measured uncertainty.

In [ ]:
EXPERIMENTS = [
    {'name': 'dr_baseline', 'seed': C.CUBE_LAYOUT_SEED, 'yaw_offset_deg': 0, 'paper_roughness': C.BACKDROP_ROUGHNESS},
    {'name': 'dr_layout_a', 'seed': 11, 'yaw_offset_deg': 0, 'paper_roughness': C.BACKDROP_ROUGHNESS},
    {'name': 'dr_layout_b', 'seed': 29, 'yaw_offset_deg': 0, 'paper_roughness': C.BACKDROP_ROUGHNESS},
    {'name': 'dr_yaw_plus10', 'seed': C.CUBE_LAYOUT_SEED, 'yaw_offset_deg': 10, 'paper_roughness': C.BACKDROP_ROUGHNESS},
    {'name': 'dr_rough_paper', 'seed': C.CUBE_LAYOUT_SEED, 'yaw_offset_deg': 0, 'paper_roughness': 0.50},
]
for e in EXPERIMENTS: print(e)

## 2. Preview layouts without Isaac

`scene_config.hackathon_cubes(seed)` is deterministic and respects the starter's placement exclusions. Preview seeds before paying the render cost. The first two photo-fitted cubes stay fixed by design.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
markers = ['o', 's', '^', 'D', 'P']
for marker, e in zip(markers, EXPERIMENTS):
    cubes = C.hackathon_cubes(e['seed'])
    ax.scatter([x for x, y, c in cubes], [y for x, y, c in cubes], label=e['name'], marker=marker, alpha=.7)
ax.scatter([C.CUBE_STACK_XY[0]], [C.CUBE_STACK_XY[1]], marker='*', s=180, color='black', label='tower site')
ax.set(xlabel='world X (m)', ylabel='world Y (m)', title='Candidate object layouts'); ax.axis('equal'); ax.legend(); plt.show()

## 3. Make isolated Isaac captures

The following cell creates a small wrapper inside `runs/_tutorial_tools`. The wrapper changes configuration **in memory** before running the existing capture script; it does not edit production calibration files. Every output receives `randomization.json`. Review the generated wrapper before executing it.

In [ ]:
TOOLS = RUNS / '_tutorial_tools'; TOOLS.mkdir(parents=True, exist_ok=True)
wrapper = TOOLS / 'randomized_capture.py'
wrapper.write_text('''import argparse, json, runpy, sys
from pathlib import Path

ap = argparse.ArgumentParser()
ap.add_argument('--sim', required=True)
ap.add_argument('--out', required=True)
ap.add_argument('--seed', type=int, required=True)
ap.add_argument('--yaw-offset-deg', type=float, default=0)
ap.add_argument('--paper-roughness', type=float, default=0.30)
a = ap.parse_args()
sim = Path(a.sim).resolve(); out = Path(a.out).resolve()
sys.path.insert(0, str(sim))
import scene_config as C
C.CUBE_LAYOUT_SEED = a.seed
C.SORT_CUBES = C.hackathon_cubes(a.seed)
C.SORT_CUBE_YAWS_DEG = [y + a.yaw_offset_deg for y in C.SORT_CUBE_YAWS_DEG]
C.BACKDROP_ROUGHNESS = a.paper_roughness
sys.argv = ['capture_scene.py', '--out', str(out), '--camera', 'table']
runpy.run_path(str(sim / 'capture_scene.py'), run_name='__main__')
(out / 'randomization.json').write_text(json.dumps(vars(a), indent=2) + '\n')
''')
print(wrapper)
print(wrapper.read_text())

In [ ]:
ISAAC = os.environ.get('ISAAC')
commands = []
for e in EXPERIMENTS:
    out = RUNS / e['name']
    cmd = [str(Path(ISAAC) / 'python.sh') if ISAAC else '$ISAAC/python.sh', str(wrapper),
           '--sim', str(SIM), '--out', str(out), '--seed', str(e['seed']),
           '--yaw-offset-deg', str(e['yaw_offset_deg']), '--paper-roughness', str(e['paper_roughness'])]
    commands.append(cmd); print(' '.join(cmd))
print('Run only into new or empty directories. Captures can take several minutes each.')

RUN_LIVE = False  # Read the commands, confirm output directories, then opt in.
if RUN_LIVE:
    if not ISAAC: raise EnvironmentError('Set ISAAC first')
    for cmd in commands:
        out = Path(cmd[cmd.index('--out') + 1])
        if out.exists() and any(out.iterdir()): raise FileExistsError(f'Not empty: {out}')
        subprocess.run(cmd, cwd=SIM, check=True)

## 4. Reject unrealistic samples

Before inference, review every render. Reject intersecting objects, objects outside the reachable/visible workspace, impossible lighting, changed semantic colours, or a background that hides task objects. Record the rejection reason; silently deleting ugly samples biases the dataset.

In [ ]:
available = [(e['name'], RUNS / e['name'] / 'world_camera.png') for e in EXPERIMENTS if (RUNS / e['name'] / 'world_camera.png').exists()]
if available:
    fig, axes = plt.subplots(len(available), 1, figsize=(12, 5*len(available)))
    axes = [axes] if len(available) == 1 else axes
    for ax, (name, path) in zip(axes, available):
        ax.imshow(plt.imread(path)); ax.set_title(name); ax.axis('off')
    plt.tight_layout(); plt.show()
else: print('No randomized captures yet.')

## 5. Run the same Cosmos and planning pipeline

For each accepted capture, use identical model weights and commands:

```bash
"$COSMOS" fanuc_er4ia_sim/perceive.py --run fanuc_er4ia_sim/runs/dr_layout_a --stack-cubes
fanuc_er4ia_sim/.venv/bin/python fanuc_er4ia_sim/plan_sort.py --run fanuc_er4ia_sim/runs/dr_layout_a
```

Use notebook 04 to compare matched cubes, worst XY/yaw error, skipped reasons, and—only after a safe plan exists—physical simulation execution. Reserve at least one seed as a held-out test; do not tune prompts against every scene.

## 6. Lighting extension

After the layout study works, inspect `cell.add_lighting()`: the dome, key, fill, and rim intensities are authored there. On an experiment branch, expose one documented `light_scale` parameter and apply the same multiplier to those four intensities. Test a small plausible range such as 0.8, 1.0, and 1.2 while holding layout fixed.

Do not silently alter the calibrated camera, semantic object colours, or geometry. Because lighting affects perception, recapture and replan every condition in a new run directory; because runtime code changed, run the host tests and record the source revision in notebook 09's manifest.

## Team exercise and checkpoint

Produce five accepted variations and at least one documented rejection. Compare baseline, layout-only, yaw-only, and appearance-only conditions. Explain which change caused the largest performance drop and propose one training/prompt/system improvement.

You are finished when every image can be traced to a seed and factor values, the held-out scenes stayed untouched during tuning, and no randomized quantity contradicts the real cell.